# CAPTOR on histology

The CONCH image encoder (its text tower is never used) and a table counted from
32,463 PathCap figure captions. The same code runs on UNI or Phikon-v2, encoders
with no text tower at all, once a table is built for them with
`scripts/build_table.py --preset pathology --encoder uni` (or `phikon_v2`).

CONCH and UNI are gated on the Hugging Face Hub. Request access on their model
pages, then run `huggingface-cli login`. CONCH also needs its package:
`pip install git+https://github.com/Mahmoodlab/CONCH.git`.

In [ ]:
# Setup for Google Colab: fetch the code, the example fields and CONCH (skipped when run locally).
# CONCH is gated: request access at https://huggingface.co/MahmoodLab/CONCH first, then log in below.
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/captor-repo"):
        !git clone -q https://github.com/EladYoshai/captor.git /content/captor-repo
        !pip install -q -e "/content/captor-repo[pathology]" git+https://github.com/Mahmoodlab/CONCH.git
    %cd /content/captor-repo/examples
    sys.path.insert(0, "/content/captor-repo")
    from huggingface_hub import notebook_login
    notebook_login()


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image

from captor import CAPTOR
from captor.viz import heatmap, legend, overlay

model = CAPTOR.from_pretrained("pathcap_conch_k80")
TERMS = ["tumor", "stroma", "lymphocyte", "necrosis", "adipose"]
fields = sorted(Path("images").glob("bcss_*.jpg"))

## Five tissues on three breast-cancer fields

Each field is 896 x 896 pixels at 0.25 microns per pixel (BCSS), read at 224
pixels on CONCH's 28 x 28 patch grid.

In [ ]:
fig, ax = plt.subplots(2, len(fields), figsize=(4.5 * len(fields), 9))
for c, path in enumerate(fields):
    field = Image.open(path).convert("RGB")
    seg = model.segment(field, TERMS)
    print(path.name, {t: round(s, 2) for t, s in seg.shares().items()})
    ax[0, c].imshow(field); ax[1, c].imshow(overlay(field, seg.labels))
for a in ax.ravel(): a.axis("off")
fig.legend(handles=[Patch(color=[v / 255 for v in col], label=t) for t, col in legend(TERMS)],
           loc="lower center", ncol=5, frameon=False);

## Terms beyond the five annotated tissues

In [ ]:
print(model.terms)
field = Image.open(fields[1]).convert("RGB")
seg = model.segment(field, ["tumor", "necrosis", "inflammatory", "vessel"])
fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
for i, t in enumerate(seg.terms):
    ax[i].imshow(Image.blend(field, heatmap(seg.scores[..., i], field.size), 0.6))
    ax[i].set_title(t); ax[i].axis("off")